### Importation des bibliothèques

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse as sp
import scipy.io
import os
import pandas as pd
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import normalized_mutual_info_score
import matplotlib.pyplot as plt
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary

### Configuration des chemins et des dossiers de travail

In [2]:
SEED = 42
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

hiddensize = 256
batchsize = 200
TOPN = 15
DATASETS = ['20NG', 'AGNews', 'IMDB', 'YahooAnswer']
KVALUES = [50, 100]

# Chemins 
DATAROOT = r"C:\Users\Home\Documents\M2\projet_PPD\data"
DVAEROOT = r"C:\Users\Home\Documents\M2\projet_PPD\output\DVAE"
os.makedirs(DVAEROOT, exist_ok=True)
print(f"DATAROOT: {DATAROOT}")
print(f"DVAEROOT: {DVAEROOT}")

Device: cuda
DATAROOT: C:\Users\Home\Documents\M2\projet_PPD\data
DVAEROOT: C:\Users\Home\Documents\M2\projet_PPD\output\DVAE


### Initialisation du seed et chargement des datasets

In [19]:
def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def load_dataset(name):
    path = os.path.join(DATAROOT, name)
    
    train_files = ['trainbow.npz', 'train.npz', 'train_bow.npz', 'trainbow.npy']
    test_files = ['testbow.npz', 'test.npz', 'test_bow.npz', 'testbow.npy']
    vocab_files = ['vocab.txt', 'vocabulary.txt']
    
    # Train
    train_file = None
    for f in train_files:
        train_path = os.path.join(path, f)
        if os.path.exists(train_path):
            train_file = train_path
            break
    
    if train_file is None or not os.path.exists(train_file):
        print(f" Aucun fichier train trouvé: {train_files}")
        print(f" Fichiers présents: {os.listdir(path)[:10]}...")
        raise FileNotFoundError(f"Pas de données train pour {name}")
    
    # Charge train
    print(f" Train: {os.path.basename(train_file)}")
    if train_file.endswith('.npz'):
        train = sp.load_npz(train_file).toarray()
    else:  # .npy
        train = np.load(train_file)
    
    # Test
    test_file = None
    for f in test_files:
        test_path = os.path.join(path, f)
        if os.path.exists(test_path):
            test_file = test_path
            break
    
    print(f" Test: {os.path.basename(test_file)}")
    if test_file.endswith('.npz'):
        test = sp.load_npz(test_file).toarray()
    else:
        test = np.load(test_file)
    
    # Vocab
    vocab_file = None
    for f in vocab_files:
        vocab_path = os.path.join(path, f)
        if os.path.exists(vocab_path):
            vocab_file = vocab_path
            break
    
    if vocab_file:
        with open(vocab_file, encoding='utf-8') as f:
            vocab = [w.strip() for w in f]
    else:
        print(" Pas de vocab.txt, on génère un dummy")
        vocab = [f"word_{i}" for i in range(train.shape[1])]
    
    return train, test, vocab

def topic_diversity(beta, vocab, topn=15):
    all_words = []
    for k in range(beta.shape[0]):
        topidx = np.argsort(beta[k, :])[-topn:]
        all_words.extend([vocab[i] for i in topidx])
    return len(set(all_words)) / len(all_words)

def purity_score(y_true, y_pred):
    y_true_int = y_true.astype(np.int64)
    y_pred_int = y_pred.astype(np.int64)
    
    purity = 0
    for c in np.unique(y_pred_int):
        idx = np.where(y_pred_int == c)[0]
        if len(idx) == 0: 
            continue
        counts = np.bincount(y_true_int[idx])
        purity += counts.max()
    return purity / len(y_true)



### Implémentation du modèle DVAE

In [4]:
class DVAEBaseline(nn.Module):
    def __init__(self, vocabsize, numtopics, hiddensize, prioralpha, dropout=0.2, klweight=1.0):
        super().__init__()
        self.klweight = klweight
        self.numtopics = numtopics
        
        self.encoder = nn.Sequential(
            nn.Linear(vocabsize, hiddensize),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hiddensize, numtopics)
        )
        
        self.beta = nn.Parameter(torch.Tensor(numtopics, vocabsize))
        nn.init.xavier_uniform_(self.beta)
        self.decoderbn = nn.BatchNorm1d(vocabsize, affine=False)
        self.register_buffer('prioralpha', torch.full((numtopics,), prioralpha))
    
    def encode(self, x):
        raw = self.encoder(x)
        alpha = F.softplus(raw + 1e-6)
        return torch.clamp(alpha, min=1e-3, max=50.0)
    
    def sample_theta(self, alpha):
        if self.training:
            concentration = torch.clamp(alpha, min=1e-2)
            rate = torch.ones_like(concentration)
            gamma = torch.distributions.Gamma(concentration, rate)
            z = gamma.rsample()
            theta = z / (z.sum(dim=1, keepdim=True) + 1e-12)
        else:
            theta = alpha / (alpha.sum(dim=1, keepdim=True) + 1e-12)
        return torch.clamp(theta, min=1e-12)
    
    def decode(self, theta):
        res = torch.matmul(theta, self.beta)
        res = self.decoderbn(res)
        return F.log_softmax(res, dim=1)
    
    def forward(self, x_raw):
        # Normalisation
        x_sum = x_raw.sum(dim=1, keepdim=True)
        x_norm = x_raw / torch.clamp(x_sum, min=1e-12)
        
        # Forward pass
        alpha = self.encode(x_norm)
        theta = self.sample_theta(alpha)
        logprobs = self.decode(theta)
        recon = -(x_raw * logprobs).sum(dim=1).mean()
        mean_alpha = alpha.mean(dim=0)  # (K,)
        prior_logit = self.prioralpha.detach().clone().to(alpha.device)  
        kl = F.kl_div(
            F.log_softmax(prior_logit.unsqueeze(0), dim=0),
            F.softmax(mean_alpha, dim=0).unsqueeze(0),
            reduction='batchmean'
        )

        
        return recon + self.klweight * kl
    
    def get_beta(self):
        self.eval()
        with torch.no_grad():
            ident = torch.eye(self.numtopics).to(self.beta.device)
            logits = self.decoderbn(torch.matmul(ident, self.beta))
            return F.softmax(logits, dim=1)
    
    def get_theta(self, x_raw):
        self.eval()
        with torch.no_grad():
            x_sum = x_raw.sum(dim=1, keepdim=True)
            x_norm = x_raw / torch.clamp(x_sum, min=1e-12)
            alpha = self.encode(x_norm)
            return alpha / (alpha.sum(dim=1, keepdim=True) + 1e-12)


### Boucle d'entraînement du modèle DVAE et calcul des métriques

In [20]:
def train_dvae_pipeline():
    SAVE_PATH = os.path.join(DVAEROOT, "DVAE_REPRO_METRICS.csv")
    results = []
    
    # 1. Charger l'existant
    if os.path.exists(SAVE_PATH):
        results = pd.read_csv(SAVE_PATH).to_dict('records')
        print(f"--- {len(results)} résultats chargés. On ne calcule que le reste. ---")

    for dataset in DATASETS:
        faits = [r['K'] for r in results if r['Dataset'] == dataset]
        if all(k in faits for k in KVALUES):
            continue

        trainbow, testbow, vocab = load_dataset(dataset)
        V = trainbow.shape[1]
        
        ytrue_path = os.path.join(DATAROOT, dataset, 'test_labels.txt')
        ytrue = np.loadtxt(ytrue_path, dtype=int) if os.path.exists(ytrue_path) else np.zeros(len(testbow))

        # Hyperparamètres par dataset
        if dataset == '20NG':
            hp_dict = {50: {'dropout': 0.98, 'klweight': 50.0, 'prioralpha': 200.0, 'epochs': 8},
                       100: {'dropout': 0.96, 'klweight': 45.0, 'prioralpha': 150.0, 'epochs': 12}}
        elif dataset == 'AGNews':
            hp_dict = {50: {'dropout': 0.92, 'klweight': 35.0, 'prioralpha': 120.0, 'epochs': 10},
                       100: {'dropout': 0.88, 'klweight': 30.0, 'prioralpha': 100.0, 'epochs': 15}}
        elif dataset == 'IMDB':
            hp_dict = {50: {'dropout': 0.99, 'klweight': 60.0, 'prioralpha': 250.0, 'epochs': 6},
                       100: {'dropout': 0.97, 'klweight': 55.0, 'prioralpha': 200.0, 'epochs': 10}}
        else:
            hp_dict = {50: {'dropout': 0.95, 'klweight': 42.0, 'prioralpha': 160.0, 'epochs': 9},
                       100: {'dropout': 0.92, 'klweight': 38.0, 'prioralpha': 130.0, 'epochs': 14}}

        for K in KVALUES:
            if any(r['Dataset'] == dataset and r['K'] == K for r in results):
                continue

            print(f" -> Entraînement DVAE {dataset} K={K}...")
            hp = hp_dict[K]
            
            set_seed(SEED)
            model = DVAEBaseline(V, K, hiddensize, hp['prioralpha'], hp['dropout'], hp['klweight']).to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

            # Train
            Xtrain = torch.tensor(trainbow, dtype=torch.float32).to(device)
            loader = DataLoader(TensorDataset(Xtrain), batch_size=batchsize, shuffle=True)
            model.train()
            for epoch in range(hp['epochs']):
                for batch in loader:
                    batch = batch[0].to(device)
                    optimizer.zero_grad()
                    model(batch).backward()
                    optimizer.step()

            # Eval
            model.eval()
            with torch.no_grad():
                beta = model.get_beta().cpu().numpy()
                theta = model.get_theta(torch.tensor(testbow, dtype=torch.float32).to(device)).cpu().numpy()
                ypred = theta.argmax(axis=1)
                
                TD = topic_diversity(beta, vocab, TOPN)
                purity = purity_score(ytrue, ypred)
                NMI = normalized_mutual_info_score(ytrue, ypred)

            # 2. Sauvegarde 
            results.append({'Dataset': dataset, 'K': K, 'TD': round(TD, 4), 
                            'Purity': round(purity, 4), 'NMI': round(NMI, 4)})
            pd.DataFrame(results).to_csv(SAVE_PATH, index=False)

    # Affichage final
    df = pd.DataFrame(results).sort_values(['Dataset', 'K'])
    print("\n" + "="*50 + "\n   SYNTHÈSE FINALE DVAE\n" + "="*50)
    print(df.to_string(index=False))
    return df

# Lancement
results_df = train_dvae_pipeline()

--- 8 résultats chargés. On ne calcule que le reste. ---

   SYNTHÈSE FINALE DVAE
    Dataset   K     TD  Purity    NMI
       20NG  50 0.7320  0.0823 0.0356
       20NG 100 0.5733  0.0535 0.0013
     AGNews  50 0.7720  0.2736 0.0228
     AGNews 100 0.6513  0.2952 0.0265
       IMDB  50 0.5613  0.5000 0.0000
       IMDB 100 0.4040  0.5445 0.0188
YahooAnswer  50 0.7467  0.1360 0.0164
YahooAnswer 100 0.6067  0.1072 0.0025


### Calcul de la Cohérence

In [12]:
print("Calcul DVAE C_V Palmetto en cours...")

DVAEROOT = r"C:\Users\Home\Documents\M2\Projet_PPD\output\DVAE"
SAVE_FILE = os.path.join(DVAEROOT, "DVAE_CV_OFFICIEL.csv")
TOPN = 15
K_VALUES = [50, 100]

# 1. Charger les résultats existants 
if os.path.exists(SAVE_FILE):
    df_exist = pd.read_csv(SAVE_FILE)
    cv_dvae_results = df_exist.to_dict('records')
else:
    cv_dvae_results = []

for dataset in tqdm(DATASETS, desc="Datasets"):
    

    tous_faits = all(any(r['Dataset'] == dataset and r['K'] == k for r in cv_dvae_results) for k in K_VALUES)
    if tous_faits:
        continue

    # Chargement 
    train_bow, test_bow, vocab = load_dataset(dataset) 
    
    # Préparation Gensim
    texts = [[vocab[i] for i, freq in enumerate(doc) if freq > 0] for doc in train_bow[:10000]]
    texts = [t[:100] for t in texts if len(t) >= 2]
    
    dictionary = Dictionary(texts)
    corpus = [dictionary.doc2bow(text) for text in texts]
    
    for K in K_VALUES:
        deja_fait = any(r['Dataset'] == dataset and r['K'] == k for r in cv_dvae_results)
        if deja_fait:
            continue

        mat_path = os.path.join(DVAEROOT, dataset, f"{dataset}_DVAE_K{K}_seed42.mat")
        if not os.path.exists(mat_path):
            continue
            
        data = scipy.io.loadmat(mat_path)
        beta_keys = ['beta', 'betas', 'topic_word', 'beta_dvae']
        beta = next((data[key] for key in beta_keys if key in data and data[key].size > 0), None)
        
        if beta is not None:
            topics = []
            for k in range(min(K, beta.shape[0])):
                # TOPN = 15
                idx = np.argsort(beta[k])[-TOPN:][::-1]
                topic_words = [vocab[i] for i in idx if i < len(vocab)]
                topic_ids = [dictionary.token2id.get(w, -1) for w in topic_words]
                valid_ids = [id_ for id_ in topic_ids if id_ >= 0][:TOPN]
                if len(valid_ids) >= 2:
                    topics.append(valid_ids)
            
            if len(topics) >= K * 0.8:
                cm = CoherenceModel(topics=topics, texts=texts, corpus=corpus, 
                                    dictionary=dictionary, coherence='c_v')
                cv_score = cm.get_coherence()
                cv_dvae_results.append({"Dataset": dataset, "K": K, "C_V": round(cv_score, 4)})
                
                # Sauvegarde 
                pd.DataFrame(cv_dvae_results).to_csv(SAVE_FILE, index=False)

# --- AFFICHAGE FINAL ---
if cv_dvae_results:
    df = pd.DataFrame(cv_dvae_results)
    print("\n" + "="*45)
    print(f"{'SYNTHÈSE DVAE - COHÉRENCE C_V':^45}")
    print("="*45)
    # Affichage du pivot (plus lisible)
    print(df.pivot(index="Dataset", columns="K", values="C_V").round(4))
    print("="*45)

Calcul DVAE C_V Palmetto en cours...


Datasets:   0%|          | 0/4 [00:00<?, ?it/s]


        SYNTHÈSE DVAE - COHÉRENCE C_V        
K               50      100
Dataset                    
20NG         0.5414  0.5334
AGNews       0.4128  0.4206
IMDB         0.3880  0.3853
YahooAnswer  0.6247  0.5830


### Boucle d'entraînement du modèle

In [14]:
paper_dvae = {
    "20NG":   {"K50": {"CV": 0.331, "TD": 0.598, "Purity": 0.087, "NMI": 0.018},
               "K100": {"CV": 0.372, "TD": 0.658, "Purity": 0.104, "NMI": 0.035}},
    "IMDB":   {"K50": {"CV": 0.294, "TD": 0.050, "Purity": 0.517, "NMI": 0.000},
               "K100": {"CV": 0.290, "TD": 0.229, "Purity": 0.525, "NMI": 0.001}},
    "Yahoo":  {"K50": {"CV": 0.338, "TD": 0.674, "Purity": 0.171, "NMI": 0.030},
               "K100": {"CV": 0.376, "TD": 0.589, "Purity": 0.202, "NMI": 0.057}},
    "AGNews": {"K50": {"CV": 0.419, "TD": 0.347, "Purity": 0.713, "NMI": 0.219},
               "K100": {"CV": 0.298, "TD": 0.113, "Purity": 0.407, "NMI": 0.030}},
}


repro_dvae = {
    "20NG":   {"K50": {"CV": 0.541, "TD": 0.732, "Purity": 0.082, "NMI": 0.036},
               "K100": {"CV": 0.533, "TD": 0.573, "Purity": 0.054, "NMI": 0.001}},
    "AGNews": {"K50": {"CV": 0.413, "TD": 0.772, "Purity": 0.274, "NMI": 0.023},
               "K100": {"CV": 0.421, "TD": 0.651, "Purity": 0.295, "NMI": 0.027}},
    "IMDB":   {"K50": {"CV": 0.388, "TD": 0.561, "Purity": 0.500, "NMI": 0.000},
               "K100": {"CV": 0.385, "TD": 0.404, "Purity": 0.545, "NMI": 0.019}},
    "Yahoo":  {"K50": {"CV": 0.625, "TD": 0.747, "Purity": 0.136, "NMI": 0.016},
               "K100": {"CV": 0.583, "TD": 0.607, "Purity": 0.107, "NMI": 0.003}},
}

data = []
metrics = ["CV", "TD", "Purity", "NMI"]

for ds in ["20NG", "IMDB", "Yahoo", "AGNews"]:
    for k_key in ["K50", "K100"]:
        row = {"Dataset": ds, "K": int(k_key[1:])}
        for m in metrics:
            p_val = paper_dvae[ds][k_key][m]
            r_val = repro_dvae[ds][k_key][m]
            row[f"{m}_Papier"] = p_val
            row[f"{m}_Repro"] = r_val
            row[f"{m}_Ecart"] = round(r_val - p_val, 3)
        data.append(row)

df = pd.DataFrame(data).set_index(["Dataset", "K"])

display(df.style
    .format("{:.3f}")
    .set_caption("DVAE — Papier vs Repro")
)